# Fine-tune SmolLM3 on Kaggle\n\nFine-tune the SmolLM3 model on your custom dataset using Kaggle's free GPU!

In [ ]:
# Install required packages\n!pip install -q transformers datasets accelerate peft bitsandbytes safetensors

In [ ]:
import os\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer\nfrom datasets import load_dataset\nprint(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## Configuration

In [ ]:
# Configuration\nMODEL_NAME = "HuggingFaceTB/SmolLM3-135M"  # Use 135M for faster training, or 360M/1.7B for better quality\nOUTPUT_DIR = "/kaggle/working/zach-smollm3-finetuned"\nDATASET_NAME = "your-dataset"  # Replace with your dataset\nTRAIN_EPOCHS = 3\nBATCH_SIZE = 4\nLEARNING_RATE = 2e-4\nMAX_seq_LENGTH = 256

## Load Model and Tokenizer

In [ ]:
# Load tokenizer\ntokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\ntokenizer.pad_token = tokenizer.eos_token\n\n# Load model with 4-bit quantization for memory efficiency\nmodel = AutoModelForCausalLM.from_pretrained(\n    MODEL_NAME,\n    device_map="auto",\n    load_in_4bit=True,\n    bnb_4bit_compute_dtype=torch.float16\n)\nprint(f"Model loaded: {MODEL_NAME}")

## Prepare Dataset

In [ ]:
# Option 1: Load from HuggingFace\n# dataset = load_dataset(DATASET_NAME)\n\n# Option 2: Create simple dataset (replace with your data)\nfrom datasets import Dataset\n\ndata = {}\ndata[\"text\"] = [\n    # Add your training data here!\n    "Your task example 1: input -> output",\n    "Your task example 2: input -> output",\n    "Your task example 3: input -> output",\n]\n\ndataset = Dataset.from_dict(data)\nprint(f"Dataset size: {len(dataset)}")

In [ ]:
# Tokenize function\ndef tokenize_function(examples):\n    return tokenizer(\n        examples[\"text\"],\n        truncation=True,\n        max_length=MAX_SEQ_LENGTH,\n        padding=\"max_length\"\n    )\n\ntokenized_dataset = dataset.map(tokenize_function, batched=True)\nprint(f"Tokenized: {len(tokenized_dataset)} samples")

## Training

In [ ]:
training_args = TrainingArguments(\n    output_dir=OUTPUT_DIR,\n    num_train_epochs=TRAIN_EPOCHS,\n    per_device_train_batch_size=BATCH_SIZE,\n    learning_rate=LEARNING_RATE,\n    logging_steps=10,\n    save_strategy=\"epoch\",\n    save_total_limit=2,\n    fp16=True,\n    gradient_accumulation_steps=4,\n    dataloader_num_workers=2,\n    report_to=\"none\"\n)\n\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=tokenized_dataset,\n)\n\nprint("Starting training...")\ntrainer.train()

## Save and Export

In [ ]:
# Save model\nmodel.save_pretrained(OUTPUT_DIR)\ntokenizer.save_pretrained(OUTPUT_DIR)\nprint(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# Convert to GGUF format (requires llama.cpp)\n# !pip install gguf\n# from transformers import AutoTokenizer\n# from gguf import *\n# (See llama.cpp docs for conversion)

In [ ]:
# Download the fine-tuned model\nfrom IPython.display import FileLink\nimport shutil\nshutil.make_archive("zach-smollm3-finetuned", 'zip', OUTPUT_DIR)\nFileLink("zach-smollm3-finetuned.zip")